# Stage 4 Detector -> VLM (Kaggle, Clean Prompt Path)

This notebook runs the full Stage 4 pipeline on predicted detector boxes with the clean `_nocroppath` Stage 3 prompt path.

`val images -> detector -> pred crops -> Stage3 VLM -> Stage4 eval -> tar.gz`

The final archive is saved to `/kaggle/working`.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import json

DATASET_ROOT_CANDIDATES = [
    Path('/kaggle/input/datasets/kostyaryazanov/idid-coco-v3'),
    Path('/kaggle/input/idid-coco-v3'),
]
JSONL_REL = Path('stage3_regrouped_v2/val/vlm_labels_v1_val_v2.annotated.jsonl')

DATA_ROOT = None
for root in DATASET_ROOT_CANDIDATES:
    if (root / JSONL_REL).exists():
        DATA_ROOT = root
        break

if DATA_ROOT is None:
    raise FileNotFoundError(
        'DATA_ROOT not found: stage3_regrouped_v2/val/vlm_labels_v1_val_v2.annotated.jsonl'
    )

REPO_DIR = Path('/kaggle/working/vlm-for-insulator-defect-detection')
REPO_URL = 'https://github.com/konstRyaz/vlm-for-insulator-defect-detection.git'
RUN_NAME = 'stage4_detector_to_vlm_pred_val_kaggle'
STAGE4_PROMPT_VERSION = 'qwen_vlm_labels_v1_prompt_v7f_flashover_unclear_to_unknown_nocroppath'

print('DATA_ROOT:', DATA_ROOT)
print('REPO_DIR :', REPO_DIR)
print('RUN_NAME :', RUN_NAME)
print('STAGE4_PROMPT_VERSION:', STAGE4_PROMPT_VERSION)


In [ ]:
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt', 'hydra-core'], check=True)

print('Repo ready:', REPO_DIR)

In [ ]:
import yaml

def pick_existing(candidates):
    for c in candidates:
        p = Path(c)
        if p.exists():
            return p
    raise FileNotFoundError("Not found among candidates:\n" + "\n".join(str(x) for x in candidates))

# Stage 3 GT labels
stage3_gt_jsonl = pick_existing([
    DATA_ROOT / 'stage3_regrouped_v2/val/vlm_labels_v1_val_v2.annotated.jsonl',
    DATA_ROOT / 'outputs/stage3_regrouped_v2/val/vlm_labels_v1_val_v2.annotated.jsonl',
    REPO_DIR / 'outputs/stage3_regrouped_v2/val/vlm_labels_v1_val_v2.annotated.jsonl',
])

# Detector assets root
DETECTOR_DATASET_BASE = Path('/kaggle/input/datasets/kostyaryazanov/idid-detector-assets')
DETECTOR_ASSETS_ROOT = (
    DETECTOR_DATASET_BASE / 'idid-detector-assets'
    if (DETECTOR_DATASET_BASE / 'idid-detector-assets').exists()
    else DETECTOR_DATASET_BASE
)

if not DETECTOR_ASSETS_ROOT.exists():
    raise FileNotFoundError(f"DETECTOR_ASSETS_ROOT not found: {DETECTOR_ASSETS_ROOT}")

detector_input_dir = DETECTOR_ASSETS_ROOT / 'data/processed/val/images'
coco_json = DETECTOR_ASSETS_ROOT / 'data/processed/val/annotations.json'
weights_path = DETECTOR_ASSETS_ROOT / 'outputs/train/detector_baseline/best.pth'
gt_jsonl = pick_existing([
    DETECTOR_ASSETS_ROOT / 'analysis/stage4_gt_remapped.jsonl',
    stage3_gt_jsonl,
])

for p in [detector_input_dir, coco_json, weights_path]:
    if not p.exists():
        raise FileNotFoundError(f"Missing path: {p}")

print('stage3_gt_jsonl     :', stage3_gt_jsonl)
print('gt_jsonl            :', gt_jsonl)
print('DETECTOR_ASSETS_ROOT:', DETECTOR_ASSETS_ROOT)
print('detector_input_dir  :', detector_input_dir)
print('coco_json           :', coco_json)
print('weights_path        :', weights_path)

ceiling_run_dir = None
ceiling_candidates = [
    DATA_ROOT / 'outputs/stage3_vlm_baseline_runs/stage3_qwen_val_v2_clean_final',
    REPO_DIR / 'outputs/stage3_vlm_baseline_runs/stage3_qwen_val_v2_clean_final',
]
for found in Path('/kaggle/input').rglob('stage3_qwen_val_v2_clean_final/predictions_vlm_labels_v1.jsonl'):
    ceiling_candidates.append(found.parent)
for c in ceiling_candidates:
    if Path(c).exists():
        ceiling_run_dir = Path(c)
        break

cfg = {
    'version': 1,
    'name': 'stage4_detector_to_vlm_pred_val_kaggle',
    'stage4': {
        'run_name': RUN_NAME,
        'split': 'val',
        'output_root': str(REPO_DIR / 'outputs/stage4'),
    },
    'detector': {
        'experiment': 'detector_baseline',
        'input_dir': str(detector_input_dir),
        'config_path': str(REPO_DIR / 'src/configs/infer.yaml'),
        'weights_path': str(weights_path),
        'conf_threshold': 0.05,
        'iou_threshold': 0.5,
        'max_detections_per_image': 100,
        'vis_samples': 8,
        'device': 'auto',
    },
    'crop_export': {
        'coco_json': str(coco_json),
        'images_dir': str(detector_input_dir),
        'include_categories': None,
        'padding_ratio': 0.15,
        'manifest_name': 'pred_manifest.jsonl',
        'summary_name': 'pred_manifest_summary.json',
        'limit': None,
    },
    'vlm': {
        'backend_mode': 'qwen_hf',
        'model_id': 'Qwen/Qwen2.5-VL-3B-Instruct',
        'prompt_version': STAGE4_PROMPT_VERSION,
        'stage3_runner_config': str(REPO_DIR / 'configs/pipeline/stage3_vlm_gt_baseline.yaml'),
        'run_id': 'stage4_pred_vlm',
    },
    'analysis': {
        'ground_truth_jsonl': str(gt_jsonl),
        'match_iou_threshold': 0.5,
        'good_crop_iou_threshold': 0.7,
    },
}

if ceiling_run_dir is not None:
    cfg['analysis']['ceiling_run_dir'] = str(ceiling_run_dir)

cfg_path = REPO_DIR / 'configs/stage4_kaggle_run.yaml'
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=False), encoding='utf-8')

print('ceiling_run_dir     :', ceiling_run_dir)
print('Config path:', cfg_path)
print(cfg_path.read_text(encoding='utf-8'))


In [ ]:
required_paths = [
    REPO_DIR / 'scripts/run_stage4_detector_to_vlm.py',
    REPO_DIR / 'configs/pipeline/stage3_vlm_gt_baseline.yaml',
    detector_input_dir,
    coco_json,
    weights_path,
    gt_jsonl,
]
for p in required_paths:
    if not Path(p).exists():
        raise FileNotFoundError(f"Missing required path: {p}")

stage3_cfg = yaml.safe_load((REPO_DIR / 'configs/pipeline/stage3_vlm_gt_baseline.yaml').read_text(encoding='utf-8'))
versions = stage3_cfg['prompt']['versions']
selected_prompt = cfg['vlm']['prompt_version']
if selected_prompt not in versions:
    raise RuntimeError(f"Prompt version missing in Stage 3 config: {selected_prompt}")
prompt_user_rel = versions[selected_prompt]['user_path']
prompt_user_abs = REPO_DIR / prompt_user_rel
if not prompt_user_abs.exists():
    raise FileNotFoundError(f"Missing selected prompt file: {prompt_user_abs}")
prompt_user_text = prompt_user_abs.read_text(encoding='utf-8')
if "crop_path" in prompt_user_text:
    raise RuntimeError("Selected Stage 4 prompt still exposes crop_path")

print('Selected clean prompt file:', prompt_user_abs)
print('Clean prompt preflight OK.')


In [ ]:
os.chdir(REPO_DIR)

run_cmd = [sys.executable, 'scripts/run_stage4_detector_to_vlm.py', '--config', str(cfg_path)]
print('Running:', ' '.join(run_cmd))

try:
    subprocess.run(run_cmd, check=True)
    print('Stage 4 run finished.')
except subprocess.CalledProcessError as exc:
    print(f"run_stage4 failed: code={exc.returncode}")
    run_root = REPO_DIR / 'outputs/stage4' / RUN_NAME
    vlm_run_id = cfg['vlm']['run_id']
    log_paths = [
        run_root / '01_detector' / 'run_detector.log',
        run_root / '02_pred_crops' / 'run_export_pred_crops.log',
        run_root / '03_vlm_pred' / 'run_stage3_pred.log',
        run_root / '04_eval' / 'run_eval_stage4.log',
    ]
    for lp in log_paths:
        if lp.exists():
            print(f"\n===== tail: {lp} =====")
            txt = lp.read_text(encoding='utf-8', errors='replace')
            print(txt[-5000:])
    pred_vlm_run_dir = run_root / '03_vlm_pred' / vlm_run_id
    pred_vlm_jsonl = pred_vlm_run_dir / 'predictions_vlm_labels_v1.jsonl'
    if not pred_vlm_jsonl.exists():
        pred_vlm_run_dir.mkdir(parents=True, exist_ok=True)
        pred_vlm_jsonl.write_text('', encoding='utf-8')
        print(f"Created empty file: {pred_vlm_jsonl}")
    eval_cmd = [
        sys.executable,
        'scripts/eval_stage4_detector_to_vlm.py',
        '--gt-jsonl', str(cfg['analysis']['ground_truth_jsonl']),
        '--pred-manifest-jsonl', str(run_root / '02_pred_crops' / cfg['crop_export']['manifest_name']),
        '--pred-vlm-run-dir', str(pred_vlm_run_dir),
        '--detector-predictions-json', str(run_root / '01_detector' / 'predictions.json'),
        '--coco-json', str(cfg['crop_export']['coco_json']),
        '--match-iou-threshold', str(cfg['analysis']['match_iou_threshold']),
        '--good-crop-iou-threshold', str(cfg['analysis']['good_crop_iou_threshold']),
        '--output-dir', str(run_root / '04_eval'),
    ]
    if 'ceiling_run_dir' in cfg['analysis']:
        eval_cmd.extend(['--ceiling-run-dir', str(cfg['analysis']['ceiling_run_dir'])])
    print('Retry eval:', ' '.join(eval_cmd))
    subprocess.run(eval_cmd, check=True)
    print('Stage 4 eval finished (fallback path).')


In [ ]:
run_root = REPO_DIR / 'outputs/stage4' / RUN_NAME
eval_dir = run_root / '04_eval'
metrics_path = eval_dir / 'stage4_metrics.json'
breakdown_path = eval_dir / 'stage4_error_breakdown.json'
summary_path = eval_dir / 'stage4_summary.md'

metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
breakdown = json.loads(breakdown_path.read_text(encoding='utf-8'))

print('=== Stage 4 rates ===')
for k, v in metrics.get('rates', {}).items():
    print(f'{k}: {v:.4f}' if isinstance(v, (int, float)) else f'{k}: {v}')

print('\n=== Error buckets (counts) ===')
for k, v in breakdown.get('counts', {}).items():
    print(f'{k}: {v}')

print('\nSummary file:', summary_path)

In [ ]:
archive_path = Path('/kaggle/working') / f'stage4_deliverables_{RUN_NAME}.tar.gz'
if archive_path.exists():
    archive_path.unlink()

subprocess.run([
    'tar', '-czf', str(archive_path),
    '-C', '/kaggle/working',
    f'vlm-for-insulator-defect-detection/outputs/stage4/{RUN_NAME}'
], check=True)

size_mb = archive_path.stat().st_size / (1024 * 1024)
print('Archive ready:', archive_path)
print(f'Archive size: {size_mb:.2f} MB')

print('\nImportant files inside run:')
for rel in [
    '01_detector/predictions.json',
    '02_pred_crops/pred_manifest.jsonl',
    '03_vlm_pred/stage4_pred_vlm/predictions_vlm_labels_v1.jsonl',
    '04_eval/stage4_metrics.json',
    '04_eval/stage4_error_breakdown.json',
    '04_eval/stage4_case_table.csv',
    '04_eval/stage4_summary.md',
    '05_compare/ceiling_vs_actual.json',
    'stage4_run_summary.json',
]:
    p = Path('/kaggle/working/vlm-for-insulator-defect-detection/outputs/stage4') / RUN_NAME / rel
    print('-', p, '| exists:', p.exists())

Run artifact path:

`/kaggle/working/stage4_deliverables_stage4_detector_to_vlm_pred_val_kaggle.tar.gz`
